# SCHISM boundaries and namelists

**Learning goals:** Configure real tidal/ocean boundary inputs and the SCHISM namelist structure.

**Prerequisites:** Lesson 4; shared SCHISM fixtures. Actual model execution remains optional.

**Execution contract:** This lesson is **configuration-only**. Documentation rendering never executes SCHISM, downloads data, or requires MPI/Docker.

## Checkpoint

By the end of this lesson, record what was configured and which steps still require a model runtime.

Previous: [journey_04_schism_forcing](../journey_04_schism_forcing/)

Next: [journey_06_schism_real_case](../journey_06_schism_real_case/)


## Why this matters: SCHISM data preparation

**Without Rompy:** preparing HYCOM boundary conditions can mean downloading a large global dataset, selecting the run period and region, interpolating onto open-boundary nodes, extracting the required variables, and writing `elev2D.th.nc` or other SCHISM files. ERA5 and tidal inputs require similarly separate preparation steps.

**With Rompy:** source objects, grid metadata, time ranges, filters, and boundary mappings are assembled into `SCHISMConfig`. Workspace generation carries out the configured cropping, interpolation, boundary extraction, and SCHISM-format conversion. The modeller still chooses appropriate datasets, variables, coordinates, numerical settings, and scientific validation checks.

The following cells show the source fields, model domain, and generated artefacts so this automation remains inspectable.


In [ ]:
import sys
from pathlib import Path

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
             if (path / "scripts" / "schism_case_data.py").is_file())
sys.path.insert(0, str(root))
from scripts.schism_case_data import ensure_schism_data

case = ensure_schism_data()
print("Fixture directory:", case)


In [ ]:
from rompy.core.source import SourceFile
from rompy_schism.data import SCHISMDataBoundary

# Boundary objects retain the source and variable/coordinate mapping.
elevation = SCHISMDataBoundary(
    id="elev2D",
    source=SourceFile(uri=case / "hycom.nc"),
    variables=["surf_el"],
    coords={"t": "time", "y": "ylat", "x": "xlon"},
)
print("Configured boundary source:", elevation.id)
print("Tidal assets:", case / "tides")


## What boundary preparation replaces

The boundary configuration below describes the source variables and coordinate mapping. Rompy uses the SCHISM mesh to sample those fields at open-boundary nodes and writes SCHISM's boundary formats; the plot makes that source-to-boundary relationship inspectable. This is the verification step for boundary extraction.


In [ ]:
import matplotlib.pyplot as plt
import xarray as xr
from rompy.core.data import DataBlob
from rompy_schism import SCHISMGrid

grid = SCHISMGrid(hgrid=DataBlob(source=case / "hgrid.gr3"), vgrid=DataBlob(source=case / "vgrid.in"), drag=1)
dataset = xr.open_dataset(case / "hycom.nc")
fig, ax = plt.subplots(figsize=(8, 5))
dataset.surf_el.isel(time=0).plot(ax=ax, cmap="BrBG")
x, y = grid.boundary_points()
ax.scatter(x, y, s=10, c="black", label="SCHISM open-boundary nodes")
ax.set_title("HYCOM source field and SCHISM boundary samples")
ax.legend(); plt.show()
dataset.close()


## Full HYCOM variable family

The HYCOM fixture contains surface elevation plus depth-resolved `water_u`, `water_v`, `temperature`, and `salinity`. These are distinct contracts: elevation is 2-D in space, while the other variables require a depth/vertical-coordinate decision before writing SCHISM 3-D boundary files.


In [ ]:
hycom = xr.open_dataset(case / "hycom.nc")
fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)
for ax, name in zip(axes.flat, ["water_u", "water_v", "temperature", "salinity"]):
    hycom[name].isel(time=0, depth=0).plot(ax=ax, cmap="coolwarm")
    ax.set_title(f"HYCOM {name} at depth={float(hycom.depth.isel(depth=0)):.1f} m")
plt.show()
print("HYCOM dimensions:", dict(hycom.sizes))
print("3-D variables:", [name for name in ["water_u", "water_v", "temperature", "salinity"] if name in hycom])
hycom.close()


In [ ]:
tide_files = sorted((case / "tides" / "oceanum-atlas").glob("*.nc"))
print("Tidal constituents available:", sorted({p.name.split("_")[1].upper() for p in tide_files if "_" in p.name}))
print("Atlas files:", len(tide_files))
print("Configured assumptions: TPXO-style atlas, M2/S2/N2 constituents, bilinear interpolation, and open-boundary extraction.")
print("Nodal corrections and tidal potential must be chosen for the simulation epoch; their presence in configuration is not a validation of tidal skill.")


### Progressive boundary configuration

The existing elevation boundary is the supported, executable path. The 3-D variables above are intentionally inspected before enabling them: the SCHISM plugin must know the expected depth convention and generated file contract. Do not infer scientific validity from a successful interpolation alone.


## Rompy processing: HYCOM to an open-boundary file

The HYCOM plots show what Rompy receives. This cell now performs the transformation: Rompy uses the SCHISM mesh and coordinate mapping to interpolate `surf_el` onto the open-boundary nodes and writes `elev2D.th.nc`.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from rompy.core.data import DataBlob
from rompy.core.source import SourceFile
from rompy.core.time import TimeRange
from rompy_schism import SCHISMGrid

processing_grid = SCHISMGrid(
    hgrid=DataBlob(source=case / "hgrid.gr3"),
    vgrid=DataBlob(source=case / "vgrid.in"),
    drag=1,
)
elevation = SCHISMDataBoundary(
    id="elev2D", source=SourceFile(uri=case / "hycom.nc"),
    variables=["surf_el"], coords={"t": "time", "y": "ylat", "x": "xlon"},
)
period = TimeRange(start="2023-01-01", end="2023-01-02", dt=3600)
with TemporaryDirectory() as output:
    generated_file = Path(elevation.get(output, grid=processing_grid, time=period))
    generated = xr.open_dataset(generated_file)
    print("Rompy generated:", generated_file.name)
    print("Generated dimensions:", dict(generated.sizes))
    print("Generated variables:", list(generated.data_vars))
    assert "time_series" in generated.data_vars
    assert generated.sizes["nOpenBndNodes"] == len(processing_grid.boundary_points()[0])
    generated.time_series.isel(time=0, nLevels=0, nComponents=0).plot(color="steelblue")
    plt.title("Rompy-generated HYCOM elevation at open-boundary nodes")
    plt.xlabel("open-boundary node index")
    plt.show()
    generated.close()
